# **Second Homework Assignment**

In [5]:
import numpy as np
import pandas as pd
import re

# Used for setting display settings to increase readability.
from IPython.core.display import display_html

In [17]:
# Custom formater.
def fromat_nan(val):
    if pd.isna(val):
        return ""
    if isinstance(val, float):
        return f"{round(val, 3)}"
    return f"{val}"

# Helper function that displays each dataframe in its own column.
def display_list(dfs, index=True, axis=-1, minmax="min"):
    # Convert each split to HTML with proper styling for side-by-side display.
    html_str = "<table><tr>" 
    for df in dfs:
        # Dataframe styling options.
        styled_df = df.style.format(fromat_nan)
        if axis >= 0 and minmax == "min":
            styled_df = styled_df.highlight_min(axis=axis, color='darkgreen')
            styled_df = styled_df.highlight_max(axis=axis, color='darkred')
        if axis >= 0 and minmax == "max":
            styled_df = styled_df.highlight_max(axis=axis, color='darkgreen')
            styled_df = styled_df.highlight_min(axis=axis, color='darkred')
        styled_html = styled_df.to_html(index=index)

        # Enables newlines and bold titles.
        name_html = df.attrs['name'].replace('\n', '<br>')
        name_html = f"<div style='text-align: left; font-weight: bold;'>{name_html}</div>"
        html_str += f"<td style='vertical-align: top; padding: 5px;'>{name_html + styled_html}</td>"
    html_str += "</tr></table>"
    display_html(html_str, raw=True)
    
# Helper function to display one data frame as multiple columns.
def display(df, cols, index=True):
    # Split the dataframe into columns.
    ratio = int(np.ceil(len(df) / cols))
    df_split = [df.iloc[i * ratio: (i + 1) * ratio] for i in range(cols)]
    display_list(df_split, index)


In [7]:
# Calculates means based on axis and returns the dataframe.
def get_means(dfs, axis):
    lst = []
    for df in dfs:
        mean = df.mean(axis).to_frame(name='Mean')
        mean.attrs['name'] = df.attrs['name']
        lst.append(mean)
    return lst

## **Time Comparisons**

In [ ]:
# Load all result CSVs and concatenate (skip results_gpu_<grid>_<block>.csv — used in the next cell only).
import glob
from pathlib import Path

methods = ["base", "opt", "novoid", "omp", "opt_gpu"]
sizes = [256, 512, 1024, 2048, 4096]

_csv = pd.concat(
    [pd.read_csv(f) for f in glob.glob("results_*.csv") if not re.fullmatch(r"results_gpu_\d+_\d+\.csv", Path(f).name)],
    ignore_index=True,
)
_csv.columns = [c.strip() for c in _csv.columns]
_csv["Time"] = pd.to_numeric(_csv["Time"], errors="coerce")

In [18]:
_means = _csv.groupby(["Size", "Method"])["Time"].mean()

df = pd.DataFrame(index=sizes, columns=methods)
df.index.name = "Size"
for method in methods:
    for size in sizes:
        key = (size, method)
        df.loc[size, method] = _means[key] if key in _means.index else np.nan

df.attrs['name'] = "Average execution time (s)"

print("Maximal values per row are displayed in red, while minimal are displayed in green.")
display_list([df], axis=1)

Maximal values per row are displayed in red, while minimal are displayed in green.


,base,opt,novoid,omp,opt_gpu
Size,,,,,
256,19.646,4.911,1.629,0.705,0.217
512,78.187,19.267,5.797,0.846,0.406
1024,312.362,76.713,22.987,2.874,0.607
2048,1248.073,307.526,91.992,10.972,2.056
4096,4996.188,1569.356,459.321,52.081,7.085


In [ ]:
from pathlib import Path

_parts = []
for _f in glob.glob("results_gpu_*_*.csv"):
    _m = re.fullmatch(r"results_gpu_(\d+)_(\d+)\.csv", Path(_f).name)
    if not _m:
        continue
    _t = pd.read_csv(_f)
    _t.columns = [c.strip() for c in _t.columns]
    _t["Time"] = pd.to_numeric(_t["Time"], errors="coerce")
    _parts.append(_t.assign(Grid=int(_m.group(1)), Block=int(_m.group(2))))
if _parts:
    _bg = pd.concat(_parts, ignore_index=True)
    df_blk = _bg.groupby(["Grid", "Block"])["Time"].mean().unstack("Block")
    df_blk = df_blk.dropna(axis=0, how="all").dropna(axis=1, how="all")
    df_blk.index.name = "Grid"
    df_blk.attrs["name"] = "Average execution time (s), GPU vs block size (mean of 5 runs)"
    display_list([df_blk], axis=1)

In [20]:
# Speed-up of GPU over each CPU method per size.
cpu_methods = ["base", "opt", "novoid", "omp"]

df_speedup = pd.DataFrame(index=sizes, columns=cpu_methods)
df_speedup.index.name = "Size"
for method in cpu_methods:
    for size in sizes:
        cpu = _means.get((size, method))
        gpu = _means.get((size, "opt_gpu"))
        if cpu is not None and gpu is not None and gpu > 0:
            df_speedup.loc[size, method] = cpu / gpu

df_speedup.attrs['name'] = "Speed-up of CPU method vs GPU implementation"

print("Maximal values per row are displayed in red, while minimal are displayed in green.")
display_list([df_speedup], axis=1, minmax="max")

Maximal values per row are displayed in red, while minimal are displayed in green.


,base,opt,novoid,omp
Size,,,,
256,90.369,22.589,7.492,3.241
512,192.389,47.408,14.265,2.081
1024,514.769,126.423,37.882,4.737
2048,607.039,149.575,44.743,5.336
4096,705.198,221.51,64.832,7.351


In [23]:
def _spd(t_num, t_den):
    if t_num is None or t_den is None or pd.isna(t_num) or pd.isna(t_den) or t_den == 0:
        return np.nan
    return t_num / t_den

# 1) opt vs base.
_d_opt = pd.DataFrame(index=sizes, columns=["base"])
_d_opt.index.name = "Size"
for size in sizes:
    tb, to = _means.get((size, "base")), _means.get((size, "opt"))
    _d_opt.loc[size, "base"] = _spd(tb, to)
_d_opt.attrs["name"] = "Speed-up: opt vs base"

# 2) novoid vs base and vs opt
_d_nv = pd.DataFrame(index=sizes, columns=["base", "opt"])
_d_nv.index.name = "Size"
for size in sizes:
    tb, to, tn = _means.get((size, "base")), _means.get((size, "opt")), _means.get((size, "novoid"))
    _d_nv.loc[size, "base"] = _spd(tb, tn)
    _d_nv.loc[size, "opt"] = _spd(to, tn)
_d_nv.attrs["name"] = "Speed-up: novoid vs base and opt"

# 3) omp vs base, opt, novoid
_d_omp = pd.DataFrame(index=sizes, columns=["base", "opt", "novoid"])
_d_omp.index.name = "Size"
for size in sizes:
    tx = _means.get((size, "omp"))
    _d_omp.loc[size, "base"] = _spd(_means.get((size, "base")), tx)
    _d_omp.loc[size, "opt"] = _spd(_means.get((size, "opt")), tx)
    _d_omp.loc[size, "novoid"] = _spd(_means.get((size, "novoid")), tx)
_d_omp.attrs["name"] = "Speed-up: omp vs base, opt, and novoid"

# 4) opt_gpu vs base, opt, novoid, omp
_d_gpu = pd.DataFrame(index=sizes, columns=["base", "opt", "novoid", "omp"])
_d_gpu.index.name = "Size"
for size in sizes:
    tg = _means.get((size, "opt_gpu"))
    _d_gpu.loc[size, "base"] = _spd(_means.get((size, "base")), tg)
    _d_gpu.loc[size, "opt"] = _spd(_means.get((size, "opt")), tg)
    _d_gpu.loc[size, "novoid"] = _spd(_means.get((size, "novoid")), tg)
    _d_gpu.loc[size, "omp"] = _spd(_means.get((size, "omp")), tg)
_d_gpu.attrs["name"] = "Speed-up: opt_gpu vs base, opt, novoid, and omp"
display_list([_d_opt, _d_nv, _d_omp, _d_gpu], axis=1, minmax="max")

Speed-up: opt vs base 
 
 
 
   
 base 
 
 
 Size 
   
 
 
 
 
 256 
 4.001 
 
 
 512 
 4.058 
 
 
 1024 
 4.072 
 
 
 2048 
 4.058 
 
 
 4096 
 3.184 
 
 
 
 Speed-up: novoid vs base and opt 
 
 
 
   
 base 
 opt 
 
 
 Size 
   
   
 
 
 
 
 256 
 12.062 
 3.015 
 
 
 512 
 13.487 
 3.323 
 
 
 1024 
 13.589 
 3.337 
 
 
 2048 
 13.567 
 3.343 
 
 
 4096 
 10.877 
 3.417 
 
 
 
 Speed-up: omp vs base, opt, and novoid 
 
 
 
   
 base 
 opt 
 novoid 
 
 
 Size 
   
   
   
 
 
 
 
 256 
 27.883 
 6.97 
 2.312 
 
 
 512 
 92.441 
 22.779 
 6.854 
 
 
 1024 
 108.67 
 26.688 
 7.997 
 
 
 2048 
 113.755 
 28.029 
 8.385 
 
 
 4096 
 95.931 
 30.133 
 8.819 
 
 
 
 Speed-up: opt_gpu vs base, opt, novoid, and omp 
 
 
 
   
 base 
 opt 
 novoid 
 omp 
 
 
 Size 
   
   
   
   
 
 
 
 
 256 
 90.369 
 22.589 
 7.492 
 3.241 
 
 
 512 
 192.389 
 47.408 
 14.265 
 2.081 
 
 
 1024 
 514.769 
 126.423 
 37.882 
 4.737 
 
 
 2048 
 607.039 
 149.575 
 44.743 
 5.336 
 
 
 4096 
 705.198 
 221.51 
 64.832 
 7.351

In [ ]:
from pathlib import Path

# Same logic as the cell above; sweep CSVs live in thread_test/.
# Filename: results_gpu_<grid_size>_<block_size>.csv (first number = grid, second = CUDA block).
_parts_tt = []
for _f in glob.glob("thread_test/results_gpu_*_*.csv"):
    _m = re.fullmatch(r"results_gpu_(\d+)_(\d+)\.csv", Path(_f).name)
    if not _m:
        continue
    _t = pd.read_csv(_f)
    _t.columns = [c.strip() for c in _t.columns]
    _t["Time"] = pd.to_numeric(_t["Time"], errors="coerce")
    _parts_tt.append(_t.assign(Grid=int(_m.group(1)), Block=int(_m.group(2))))
if _parts_tt:
    _bg_tt = pd.concat(_parts_tt, ignore_index=True)
    df_blk_tt = _bg_tt.groupby(["Grid", "Block"])["Time"].mean().unstack("Block")
    df_blk_tt = df_blk_tt.dropna(axis=0, how="all").dropna(axis=1, how="all")
    df_blk_tt.index.name = "Grid size"
    df_blk_tt.columns.name = "Block size"
    df_blk_tt.attrs["name"] = "Average execution time (s), GPU vs block size (mean of 5 runs), thread_test/"
    display_list([df_blk_tt], axis=1)

In [25]:
from pathlib import Path

# Same logic as the cell above; sweep CSVs live in thread_test/.
# Filename: results_gpu_<grid_size>_<block_size>.csv (first number = grid, second = CUDA block).
_parts_tt = []
for _f in glob.glob("thread_test/results_gpu_*_*.csv"):
    _m = re.fullmatch(r"results_gpu_(\d+)_(\d+)\.csv", Path(_f).name)
    if not _m:
        continue
    _t = pd.read_csv(_f)
    _t.columns = [c.strip() for c in _t.columns]
    _t["Time"] = pd.to_numeric(_t["Time"], errors="coerce")
    _parts_tt.append(_t.assign(Grid=int(_m.group(1)), Block=int(_m.group(2))))
if _parts_tt:
    _bg_tt = pd.concat(_parts_tt, ignore_index=True)
    df_blk_tt = _bg_tt.groupby(["Grid", "Block"])["Time"].mean().unstack("Block")
    df_blk_tt = df_blk_tt.dropna(axis=0, how="all").dropna(axis=1, how="all")
    df_blk_tt.index.name = "Grid size"
    df_blk_tt.columns.name = "Block size"
    df_blk_tt.attrs["name"] = "Average execution time (s)per different block sizes"
    display_list([df_blk_tt], axis=1)

Block size,8,16,32
Grid size,,,
256,3.159,3.378,2.68
512,3.681,4.005,3.42
1024,5.25,4.519,4.591
2048,8.746,8.808,7.853
4096,15.748,15.326,17.416
